# Re-evaluation: correct zero-shot reference + per-group slices

Nothing is retrained here. Six checkpoints already exist (two arms x three
seeds) and this notebook only scores them again, fixing one staging mistake
and adding the per-group breakdown.

**What went wrong the first time.** Zero-shot recall is measured against the
set of subject-predicate-object types *seen during training*, which the
framework reads from whatever is currently staged as the train split. The
replication notebook staged the test annotations before evaluating but left
the train split holding the auto arm's labels, so all four evaluations used
its 213 seen types rather than the human arm's 94. With 213 types seen,
almost no test triplet qualifies as unseen and every zR came back 0.000.
R@100, mR@100 and F1@100 never consult training statistics, so those numbers
were and remain valid.

**The fix.** Stage the human train split once, before any evaluation, and
leave it there. Both arms are then scored against the same fixed reference:
"triplet types the human annotation never contained". That is the comparison
the original run made (its logs show 94 seen triplets for both arms) and the
only one under which the two arms' zero-shot numbers mean the same thing.

**Also added.** Each annotator group in the test split is scored separately.
Group 7 is the only test annotator with no measured convention defect, so its
margin is the one claim in chapter 6 that most needs a spread rather than a
single run.

## Before running

1. Accelerator: **GPU T4 x2**.
2. Add Input: dataset **shah9212/spatial-sgg**.
3. Add Input: committed output of the **original** training notebook
   (`notebook-ssg`) - supplies the seed-42 checkpoints.
4. Add Input: committed output of the **seed replication** notebook -
   supplies the seed-43 and seed-44 checkpoints.
5. Add Input: committed output of the **vision-language arm** notebook -
   supplies react_vlm_s42/43/44. Omit it and the run still succeeds, simply
   without that arm; the assertion below only requires four runs.
5. **Save Version > Save & Run All (Commit)**.

Runtime ~40 minutes for two arms, ~60 with the vision-language arm
(9 runs x 4 slices = 36 evaluations, no training).


## Cell 1 - install

In [1]:
%%bash
cd /kaggle/working && rm -rf SGG-Benchmark
git clone -q https://github.com/Maelic/SGG-Benchmark.git
cd SGG-Benchmark && pip install -e . -q
pip install -q ultralytics hydra-core omegaconf
echo "INSTALL DONE"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 10.1 MB/s eta 0:00:00
INSTALL DONE


## Cell 2 - copy dataset and config

In [2]:
%%bash
set -e
BASE=/kaggle/working/SGG-Benchmark
INPUT=$(dirname $(find /kaggle/input -name spatial_sgg_react.yaml | head -1))
echo "found data at: $INPUT"
mkdir -p $BASE/datasets $BASE/configs/hydra/Spatial
cp -r $INPUT/spatial_sgg $BASE/datasets/
cp -r $INPUT/spatial_sgg_yolo $BASE/datasets/
cp $INPUT/spatial_sgg_react.yaml $BASE/configs/hydra/Spatial/react.yaml
echo "DATA COPIED"


found data at: /kaggle/input/datasets/shah9212/spatial-sgg
DATA COPIED


## Cell 3 - background-index patch (idempotent)

In [3]:
import json, glob, os
os.chdir("/kaggle/working/SGG-Benchmark")
for p in sorted(glob.glob("datasets/spatial_sgg/*/_annotations.human.coco.json") +
                glob.glob("datasets/spatial_sgg/*/_annotations.auto.coco.json")):
    d = json.load(open(p))
    if any(c["name"] == "__background__" for c in d["categories"]):
        continue
    for c in d["categories"]:      c["id"] += 1
    for c in d["rel_categories"]:  c["id"] += 1
    for a in d["annotations"]:     a["category_id"]  += 1
    for r in d["rel_annotations"]: r["predicate_id"] += 1
    d["categories"].insert(0, {"id": 0, "name": "__background__", "supercategory": "none"})
    d["rel_categories"].insert(0, {"id": 0, "name": "__no_relation__"})
    json.dump(d, open(p, "w"))
print("PATCH OK")


PATCH OK


## Cell 4 - gather the six checkpoints and the detector\n\nTwo arms x three seeds. Seed 42 comes from the original training notebook, seeds 43 and 44 from the replication; both must be attached as inputs.

In [4]:
import glob, os, shutil
os.chdir("/kaggle/working/SGG-Benchmark")

det = glob.glob("/kaggle/input/**/yolov8m_spatial.pt", recursive=True)
assert det, "detector not found - attach the original training notebook output"
os.makedirs("checkpoints/BACKBONES", exist_ok=True)
shutil.copy(det[0], "checkpoints/BACKBONES/yolov8m_spatial.pt")
print("detector:", det[0])

# seed 42 folders are named react_human / react_auto; 43 and 44 carry _sNN
WANT = {"react_human": ("human", 42), "react_auto": ("auto", 42),
        "react_human_s43": ("human", 43), "react_auto_s43": ("auto", 43),
        "react_human_s44": ("human", 44), "react_auto_s44": ("auto", 44),
        # the vision-language arm, trained in its own notebook; every seed
        # carries an _sNN suffix because it never had a seed-42-only run
        "react_vlm_s42": ("vlm", 42), "react_vlm_s43": ("vlm", 43),
        "react_vlm_s44": ("vlm", 44)}

RUNS = {}
for name, (arm, seed) in WANT.items():
    cfgs = [p for p in glob.glob(f"/kaggle/input/**/{name}/hydra_config.yaml", recursive=True)
            if os.path.basename(os.path.dirname(p)) == name]
    ckpts = [p for p in glob.glob(f"/kaggle/input/**/{name}/best_model_epoch_*.pth", recursive=True)
             if os.path.basename(os.path.dirname(p)) == name]
    if not (cfgs and ckpts):
        print(f"  MISSING {name}: cfg={len(cfgs)} ckpt={len(ckpts)}")
        continue
    RUNS[name] = {"arm": arm, "seed": seed,
                  "cfg": sorted(cfgs)[-1], "ckpt": sorted(ckpts)[-1]}
    print(f"  found {name}: {os.path.basename(RUNS[name]['ckpt'])}")

print(f"\n{len(RUNS)} of 6 runs available")
assert len(RUNS) >= 4, "attach both training notebooks' outputs as inputs"


detector: /kaggle/input/notebooks/shah9212/notebook-ssg/SGG-Benchmark/checkpoints/BACKBONES/yolov8m_spatial.pt
  found react_human: best_model_epoch_12.pth
  found react_auto: best_model_epoch_18.pth
  found react_human_s43: best_model_epoch_8.pth
  found react_auto_s43: best_model_epoch_24.pth
  found react_human_s44: best_model_epoch_6.pth
  found react_auto_s44: best_model_epoch_8.pth
  found react_vlm_s42: best_model_epoch_20.pth
  found react_vlm_s43: best_model_epoch_23.pth
  found react_vlm_s44: best_model_epoch_14.pth

9 of 6 runs available


## Cell 5 - stage the fixed zero-shot reference, and build per-group test sets\n\nThe train split is staged to HUMAN and left alone for the rest of the notebook: that is what defines the zero-shot set for every arm. The per-group test files filter images by the `group_N_` filename prefix, carrying their annotations and relations with them.

In [5]:
import glob, os, shutil
os.chdir("/kaggle/working/SGG-Benchmark")

det = glob.glob("/kaggle/input/**/yolov8m_spatial.pt", recursive=True)
assert det, "detector not found - attach the original training notebook output"
os.makedirs("checkpoints/BACKBONES", exist_ok=True)
shutil.copy(det[0], "checkpoints/BACKBONES/yolov8m_spatial.pt")
print("detector:", det[0])

WANT = {"react_human": ("human", 42), "react_auto": ("auto", 42),
        "react_human_s43": ("human", 43), "react_auto_s43": ("auto", 43),
        "react_human_s44": ("human", 44), "react_auto_s44": ("auto", 44),
        "react_vlm_s42": ("vlm", 42), "react_vlm_s43": ("vlm", 43),
        "react_vlm_s44": ("vlm", 44)}

RUNS = {}
for name, (arm, seed) in WANT.items():
    cfgs = [p for p in glob.glob(f"/kaggle/input/**/{name}/hydra_config.yaml", recursive=True)
            if os.path.basename(os.path.dirname(p)) == name]
    ckpts = [p for p in glob.glob(f"/kaggle/input/**/{name}/best_model_epoch_*.pth", recursive=True)
             if os.path.basename(os.path.dirname(p)) == name]
    if not (cfgs and ckpts):
        print(f"  MISSING {name}: cfg={len(cfgs)} ckpt={len(ckpts)}")
        continue
    RUNS[name] = {"arm": arm, "seed": seed,
                  "cfg": sorted(cfgs)[-1], "ckpt": sorted(ckpts)[-1]}
    print(f"  found {name}: {os.path.basename(RUNS[name]['ckpt'])}")

print(f"\n{len(RUNS)} of {len(WANT)} runs available")
assert len(RUNS) >= 4, "attach the training notebooks' outputs as inputs"
missing_arms = sorted({a for a, _ in WANT.values()} - {r["arm"] for r in RUNS.values()})
if missing_arms:
    print(f"NOTE: no checkpoints found for {missing_arms}. That arm will be "
          f"absent from the output rather than wrong in it; attach its "
          f"notebook output if you meant to include it.")

detector: /kaggle/input/notebooks/shah9212/notebook-ssg/SGG-Benchmark/checkpoints/BACKBONES/yolov8m_spatial.pt
  found react_human: best_model_epoch_12.pth
  found react_auto: best_model_epoch_18.pth
  found react_human_s43: best_model_epoch_8.pth
  found react_auto_s43: best_model_epoch_24.pth
  found react_human_s44: best_model_epoch_6.pth
  found react_auto_s44: best_model_epoch_8.pth
  found react_vlm_s42: best_model_epoch_20.pth
  found react_vlm_s43: best_model_epoch_23.pth
  found react_vlm_s44: best_model_epoch_14.pth

9 of 9 runs available


## Cell 6 - evaluate every run on every slice\n\n`--eval-only` is a no-op with this config, so `inference()` is called directly. Results are collected into one dictionary and printed as JSON at the end for easy copy-out.

In [6]:
import json, os, shutil
os.chdir("/kaggle/working/SGG-Benchmark")

for split in ["train", "val"]:
    shutil.copy(f"datasets/spatial_sgg/{split}/_annotations.human.coco.json",
                f"datasets/spatial_sgg/{split}/_annotations.coco.json")
print("train/val staged to HUMAN (defines the seen-triplet set; expect 94)")

TEST = "datasets/spatial_sgg/test"
full = json.load(open(f"{TEST}/_annotations.human.coco.json"))
shutil.copy(f"{TEST}/_annotations.human.coco.json", f"{TEST}/_annotations.full.coco.json")

def subset(group):
    keep = {im["id"] for im in full["images"] if im["file_name"].startswith(group + "_")}
    ann  = [a for a in full["annotations"] if a["image_id"] in keep]
    akeep = {a["id"] for a in ann}
    rel  = [r for r in full["rel_annotations"]
            if r["subject_id"] in akeep and r["object_id"] in akeep]
    d = dict(full)
    d["images"]          = [im for im in full["images"] if im["id"] in keep]
    d["annotations"]     = ann
    d["rel_annotations"] = rel
    path = f"{TEST}/_annotations.{group}.coco.json"
    json.dump(d, open(path, "w"))
    print(f"  {group}: {len(d['images'])} images, {len(rel)} relations -> {path}")
    return path

SLICES = {"full": f"{TEST}/_annotations.full.coco.json"}
for g in ["group_6", "group_7", "group_8"]:
    SLICES[g] = subset(g)

train/val staged to HUMAN (defines the seen-triplet set; expect 94)
  group_6: 100 images, 970 relations -> datasets/spatial_sgg/test/_annotations.group_6.coco.json
  group_7: 99 images, 796 relations -> datasets/spatial_sgg/test/_annotations.group_7.coco.json
  group_8: 37 images, 1052 relations -> datasets/spatial_sgg/test/_annotations.group_8.coco.json


## Cell 7 - bundle for download

In [7]:
import os, sys, json, glob, shutil, pathlib, torch, logging, statistics as st
os.chdir("/kaggle/working/SGG-Benchmark")

# --- re-apply the ultralytics patch, in case cell 1 re-cloned over it -------
_root = pathlib.Path("/kaggle/working/SGG-Benchmark/sgg_benchmark")
_old = "from ultralytics.utils.plotting import feature_visualization"
_marker = "feature_visualization = None  # removed in newer ultralytics"
_NL = chr(10)
_new = _NL.join([
    "try:",
    "    from ultralytics.utils.plotting import feature_visualization",
    "except ImportError:",
    "    feature_visualization = None  # removed in newer ultralytics; only",
    "    # used by an optional debug path this run never enables",
])
_patched = []
for _f in _root.rglob("*.py"):
    _s = _f.read_text(encoding="utf-8")
    if _marker in _s:
        continue
    if _old in _s:
        _f.write_text(_s.replace(_old, _new, 1), encoding="utf-8")
        _patched.append(str(_f.relative_to(_root)))
print("patched now:", _patched or "nothing (already patched)")

# a failed import leaves half-built modules cached; drop them or the retry
# fails on the stale entry rather than on the real code
for _m in [m for m in list(sys.modules) if m.startswith("sgg_benchmark")]:
    del sys.modules[_m]
print("cleared cached sgg_benchmark modules")

# --- now the real work -----------------------------------------------------
from omegaconf import OmegaConf
from sgg_benchmark.modeling.detector import build_detection_model
from sgg_benchmark.utils.checkpoint import DetectronCheckpointer
from sgg_benchmark.data import make_data_loader
from sgg_benchmark.engine.inference import inference

try:
    from sgg_benchmark.utils.logger import setup_logger
    logger = setup_logger("sgg_benchmark", ".", 0, verbose="INFO", steps=True)
except Exception:
    logging.basicConfig(level=logging.INFO)
    logger = logging.getLogger("sgg_benchmark")

TEST = "datasets/spatial_sgg/test"
RESULTS = {}

for name, info in RUNS.items():
    for slice_name, slice_path in SLICES.items():
        shutil.copy(slice_path, f"{TEST}/_annotations.coco.json")
        out = f"./checkpoints/spatial/re_{name}_{slice_name}"
        os.makedirs(out, exist_ok=True)
        cfg = OmegaConf.load(info["cfg"])
        cfg.output_dir = out

        model = build_detection_model(cfg).to(cfg.model.device)
        DetectronCheckpointer(cfg, model).load(info["ckpt"])
        model.eval()
        loader = make_data_loader(cfg, mode="test")[0]
        print("=" * 78, flush=True)
        print(f"EVAL {name} on {slice_name} ({len(loader.dataset)} images)", flush=True)
        print("=" * 78, flush=True)
        with torch.no_grad():
            inference(cfg, model, loader, dataset_name="SpatialRobot_test",
                      iou_types=("bbox", "relations"), box_only=False,
                      device=cfg.model.device, expected_results=[],
                      expected_results_sigma_tol=4, output_folder=out, logger=logger)

        f = f"{out}/eval_results_top_100.json"
        if os.path.exists(f):
            d = json.load(open(f))
            zs = d.get("sgdet_zeroshot_recall", {}).get("100", [])
            RESULTS[f"{name}|{slice_name}"] = {
                "arm": info["arm"], "seed": info["seed"], "slice": slice_name,
                "R@100":  st.mean(d["sgdet_recall"]["100"]) if d.get("sgdet_recall") else None,
                "mR@100": d.get("sgdet_mean_recall", {}).get("100"),
                "F1@100": d.get("sgdet_f1_score", {}).get("100"),
                "zR@100": (st.mean(zs) if zs else 0.0),
                "n_zeroshot": len(zs),
            }

json.dump(RESULTS, open("/kaggle/working/reeval_results.json", "w"), indent=1)
print("\n\n===== RESULTS =====")
print(json.dumps(RESULTS, indent=1))

patched now: ['modeling/backbone/yolo.py', 'modeling/backbone/yoloe.py']
cleared cached sgg_benchmark modules
Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Overriding model.yaml nc=80 with nc=6

                   from  n    params  module                                       arguments                     
  0                  -1  1      1392  ultralytics.nn.modules.conv.Conv             [3, 48, 3, 2]                 
  1                  -1  1     41664  ultralytics.nn.modules.conv.Conv             [48, 96, 3, 2]                
  2                  -1  2    111360  ultralytics.nn.modules.block.C2f             [96, 96, 2, True]             
  3                  -1  1    166272  ultralytics.nn.modules.conv.Conv             [

glove.6B.200d: 100%|█████████▉| 862M/862M [03:07<00:00, 4.49MB/s]

extracting word vectors into ./datasets/


glove.6B.200d: 862MB [03:22, 4.26MB/s]                           


INFO File not found:  ./datasets/glove.6B.200d.pt


...

2026-08-04 04:46:39,712 sgg_benchmark INFO: Total run time: 0:00:01 (39.42648397909628 ms / img per device, on 1 devices)
2026-08-04 04:46:39,713 sgg_benchmark INFO: Average latency per image: 39.42648397909628ms


2026-08-04 04:46:39,715 sgg_benchmark INFO: Standard deviation of latency: 6.474486767840178ms
2026-08-04 04:46:39,736 sgg_benchmark.data.build INFO: ----------------------------------------------------------------------------------------------------
2026-08-04 04:46:39,737 sgg_benchmark.data.build INFO: get dataset statistics...
2026-08-04 04:46:39,738 sgg_benchmark.data.build INFO: Loading data statistics from: ./checkpoints/spatial/re_react_vlm_s44_group_8/SpatialRobot_statistics.cache
2026-08-04 04:46:39,739 sgg_benchmark.data.build INFO: ----------------------------------------------------------------------------------------------------
2026-08-04 04:46:39,743 sgg_benchmark INFO: Dynamically loaded 94 seen triplets from training statistics for zero-shot evaluation.
creating index...
index created!
Loading and preparing results...
Converting ndarray to lists...
(1587, 7)
0/1587
DONE (t=0.01s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type 

SGG Eval: 100%|██████████| 37/37 [00:00<00:00, 121.91it/s]

2026-08-04 04:46:40,340 sgg_benchmark INFO: 
Detection evaluation mAp=0.7364
SGG eval:     R @ 20: 0.0708;     R @ 50: 0.0884;     R @ 100: 0.1093;  for mode=sgdet.
SGG eval:    mR @ 20: 0.0934;    mR @ 50: 0.1117;    mR @ 100: 0.1345;  for mode=sgdet.
----------------------- Details ------------------------
(on:0.0000) (under:0.5721) (to the left of:0.0982) (to the right of:0.1323) (in front of:0.0395) (behind:0.0225) (near:0.0766) 
--------------------------------------------------------
SGG eval:     zR @ 20: 0.0000;     zR @ 50: 0.0224;     zR @ 100: 0.0551;  for mode=sgdet.
SGG eval:     F1 @ 20: 0.0806;     F1 @ 50: 0.0986;     F1 @ 100: 0.1206;  for mode=sgdet.





===== RESULTS =====
{
 "react_human|full": {
  "arm": "human",
  "seed": 42,
  "slice": "full",
  "R@100": 0.3488657979467249,
  "mR@100": 0.3469540355775945,
  "F1@100": 0.34790729048220914,
  "zR@100": 0.004,
  "n_zeroshot": 125
 },
 "react_human|group_6": {
  "arm": "human",
  "seed": 42,
  "slice": "group_6",
  "R@100": 0.4115970418470418,
  "mR@100": 0.3818792345192731,
  "F1@100": 0.39618163252654554,
  "zR@100": 0.0,
  "n_zeroshot": 33
 },
 "react_human|group_7": {
  "arm": "human",
  "seed": 42,
  "slice": "group_7",
  "R@100": 0.33831595249965407,
  "mR@100": 0.32283675138862045,
  "F1@100": 0.3303951489742132,
  "zR@100": 0.0078125,
  "n_zeroshot": 64
 },
 "react_human|group_8": {
  "arm": "human",
  "seed": 42,
  "slice": "group_8",
  "R@100": 0.20020967857780003,
  "mR@100": 0.19686266574021674,
  "F1@100": 0.19852206579388973,
  "zR@100": 0.0,
  "n_zeroshot": 28
 },
 "react_auto|full": {
  "arm": "auto",
  "seed": 42,
  "slice": "full",
  "R@100": 0.26024714497133566,
  

## After the run\n\nDownload `reeval_results.json` (and the zip for the logs) from the committed version's Output tab, drop the json into `outputs/sgg_benchmark/`, and run `python eval/seed_stats.py`.\n\nSanity check while it runs: the log should say **94 seen triplets** for every evaluation. If it still says 213, the train split was not staged and the zero-shot numbers are again not comparable.

In [8]:
%%bash
cd /kaggle/working/SGG-Benchmark
zip -rq /kaggle/working/reeval_results.zip checkpoints/spatial -x "*.pth" -x "*.pt"
cp /kaggle/working/reeval_results.json /kaggle/working/reeval_results_copy.json
echo "-> /kaggle/working/reeval_results.zip  and  reeval_results.json"
ls -la /kaggle/working/reeval_results.*

-> /kaggle/working/reeval_results.zip  and  reeval_results.json
-rw-r--r-- 1 root root   8170 Aug  4 04:46 /kaggle/working/reeval_results.json
-rw-r--r-- 1 root root 406551 Aug  4 04:46 /kaggle/working/reeval_results.zip
